# Real-Time Use Case 1 — E-commerce Order Resolution Agent

## What this demonstrates

This notebook demonstrates a **single AI agent** built with `langchain_openai`.

The agent receives a customer request and independently decides which tool it should use.

Examples:

- "Where is order ORD1001?"
- "Can I return ORD1002?"
- "What should I do if my product is damaged?"
- "How much did I pay for ORD1010?"

## Why this is an Agent

There is **one reasoning agent** and multiple tools.

The agent can:

1. Understand the user's intent.
2. Decide whether it needs a tool.
3. Select the correct tool.
4. Pass the correct arguments.
5. Read the tool result.
6. Generate a user-friendly answer.

The workflow is:

**User → Agent → Tool selection → Tool execution → Agent response**

This is different from Agentic AI because we are not using multiple specialized agents working together.

## Step 1 — Install the required packages

We use:

- `langchain` for agent creation.
- `langchain-openai` for the OpenAI model integration.
- `python-dotenv` to load the API key from a `.env` file.
- `pandas` to read the order dataset.

Your `.env` file should contain:

```text
OPENAI_API_KEY=your_key_here
```

Restart the kernel after installation if required.

In [1]:
# Run once if required
# %pip install -U langchain langchain-openai python-dotenv pandas

## Step 2 — Import libraries and load environment variables

`load_dotenv()` reads values from the local `.env` file.

We do not use `getpass`, because the API key is expected to be stored in `.env`.

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY was not found in the .env file.")

## Step 3 — Load the order dataset

The CSV file is expected to be in the **same folder as the notebook**.

Each row represents a customer order. The tools will retrieve information from this file.

In [3]:
df = pd.read_csv("agent_ecommerce_orders.csv")
display(df.head())
print("Rows:", len(df))

,order_id,customer_name,product,status,order_date,expected_delivery,amount_inr,payment_method,issue_reported,issue_details
0,ORD1001,Asha,Wireless Headphones,Shipped,2026-09-09,2026-09-12,2499,Card,No,NaN
1,ORD1002,Ravi,Smart Watch,Delivered,2026-09-04,2026-09-07,3999,UPI,Yes,Damaged screen
2,ORD1003,Meena,Laptop Stand,Processing,2026-09-10,2026-09-14,1299,Card,No,NaN
3,ORD1004,Kiran,Mechanical Keyboard,Delivered,2026-09-01,2026-09-05,4599,NetBanking,Yes,Wrong item
4,ORD1005,Priya,USB-C Hub,Shipped,2026-09-08,2026-09-11,1899,UPI,No,NaN


Rows: 20


## Step 4 — Create the tools

A tool is a normal Python function that the agent is allowed to call.

We create four tools:

- `get_order_status` — retrieves delivery/order status.
- `get_order_amount` — retrieves payment amount and method.
- `check_return_eligibility` — applies a simple return rule.
- `get_issue_details` — checks whether an issue has already been reported.

The LLM does not directly inspect the DataFrame. Instead, it calls these controlled functions.

In [4]:
@tool
def get_order_status(order_id: str) -> str:
    """Get the current status and expected delivery date for an order ID."""
    row = df[df["order_id"].str.upper() == order_id.upper()]
    if row.empty:
        return f"Order {order_id} was not found."

    r = row.iloc[0]
    return (
        f"Order {r['order_id']} for {r['product']} has status '{r['status']}'. "
        f"Expected delivery: {r['expected_delivery']}."
    )


@tool
def get_order_amount(order_id: str) -> str:
    """Get the purchase amount and payment method for an order ID."""
    row = df[df["order_id"].str.upper() == order_id.upper()]
    if row.empty:
        return f"Order {order_id} was not found."

    r = row.iloc[0]
    return (
        f"Order {r['order_id']} amount is INR {r['amount_inr']} "
        f"paid using {r['payment_method']}."
    )


@tool
def check_return_eligibility(order_id: str) -> str:
    """Check whether a delivered order is eligible for return using a 10-day demo policy."""
    row = df[df["order_id"].str.upper() == order_id.upper()]
    if row.empty:
        return f"Order {order_id} was not found."

    r = row.iloc[0]

    if r["status"] != "Delivered":
        return f"Order {r['order_id']} is not delivered, so return eligibility cannot yet be applied."

    delivery_date = pd.to_datetime(r["expected_delivery"], errors="coerce")
    reference_date = pd.Timestamp("2026-09-10")

    if pd.isna(delivery_date):
        return "Delivery date is unavailable."

    days = (reference_date - delivery_date).days

    if days <= 10:
        return f"Eligible for return in this demo policy. Delivered {days} day(s) ago."
    return f"Not eligible under the 10-day demo return policy. Delivered {days} day(s) ago."


@tool
def get_issue_details(order_id: str) -> str:
    """Check whether an issue has been reported for an order and return its details."""
    row = df[df["order_id"].str.upper() == order_id.upper()]
    if row.empty:
        return f"Order {order_id} was not found."

    r = row.iloc[0]
    if str(r["issue_reported"]).lower() == "yes":
        return f"Issue reported for {r['order_id']}: {r['issue_details']}."
    return f"No issue is currently recorded for {r['order_id']}." 

## Step 5 — Create the OpenAI model

`ChatOpenAI` is provided by `langchain_openai`.

A low temperature is used because this is an operational support use case. We want stable and consistent behavior.

In [5]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

## Step 6 — Create the agent

The system instruction defines the agent's role and boundaries.

Important behavior:

- Use tools when order-specific information is required.
- Never invent an order status.
- Ask for an order ID when it is missing.
- Keep responses concise and customer-friendly.

In [6]:
tools = [
    get_order_status,
    get_order_amount,
    check_return_eligibility,
    get_issue_details
]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are an ecommerce customer-support agent.

Rules:
1. Use the provided tools for order-specific facts.
2. Never invent order details.
3. If the user asks about return eligibility, use the return tool.
4. If the user asks about a reported problem, use the issue tool.
5. Keep the final answer clear, concise, and customer-friendly.
"""
)

## Step 7 — Run a real-time customer query

The customer message is passed to the agent.

The agent decides by itself which tool or tools are needed.

In [7]:
question = "My order ORD1002 arrived damaged. Can I return it and what issue is recorded?"

result = agent.invoke({
    "messages": [
        {"role": "user", "content": question}
    ]
})

print(result["messages"][-1].content)

Your order ORD1002 has a reported issue of a damaged screen. It is eligible for return as it was delivered 3 days ago. If you would like, I can assist you with the return process.


## Step 8 — Try more real-time requests

These examples help demonstrate tool selection to participants.

Notice that different questions should trigger different tools.

In [8]:
test_questions = [
    "Where is ORD1001?",
    "How much did I pay for ORD1010?",
    "Is ORD1009 eligible for a return?",
    "Was any issue reported for ORD1013?"
]

for q in test_questions:
    result = agent.invoke({"messages": [{"role": "user", "content": q}]})
    print("\nUSER:", q)
    print("AGENT:", result["messages"][-1].content)


USER: Where is ORD1001?
AGENT: Your order ORD1001 for Wireless Headphones has been shipped and is expected to be delivered on 2026-09-12.

USER: How much did I pay for ORD1010?
AGENT: You paid INR 14999 for order ORD1010 using NetBanking.

USER: Is ORD1009 eligible for a return?
AGENT: Yes, your order ORD1009 is eligible for return as it was delivered 4 days ago and falls within the 10-day return policy. If you need assistance with the return process, please let me know!

USER: Was any issue reported for ORD1013?
AGENT: An issue has been reported for order ORD1013: the wheel is missing. If you need assistance with resolving this, please let me know!


## What participants should observe

While demonstrating, focus on these points:

1. The user does not call a Python function directly.
2. The LLM understands the request.
3. The LLM selects the appropriate tool.
4. Tool descriptions are important because they help the model choose correctly.
5. The tool returns factual data.
6. The LLM converts that data into a natural-language response.
7. One question can cause more than one tool call.
8. The agent should not invent unavailable order information.
9. This pattern can be extended to APIs, databases, CRM systems, ticketing systems, and payment systems.
10. This is a good example of a **single-agent tool-using application**.

## Production improvements

For a real production system, add:

- Customer authentication.
- API/database access instead of a local CSV.
- PII masking.
- Tool authorization.
- Audit logging.
- Rate limiting.
- Human escalation.
- Evaluation metrics.
- Observability using LangSmith or Langfuse.
